# ArtBench — Pixel Diffusion and Latent Diffusion with Multiple Seeds

Este notebook:
- usa o **subset de 20%** do ArtBench-10;
- treina **pixel diffusion** e **latent diffusion**;
- corre automaticamente **várias seeds**;
- guarda **checkpoints**, **grids** e **métricas** por seed;
- cria ficheiros finais com **média ± desvio padrão**.

Está alinhado com o enunciado, que pede avaliação com várias seeds e reporte estatístico das métricas. fileciteturn0file0

## 1. Imports

In [1]:
from __future__ import annotations

import sys
import csv
import math
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid, save_image

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

## 2. Configuração

In [2]:
SEEDS = [2,22]

IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 0

PIXEL_EPOCHS = 50
AE_EPOCHS = 20
LATENT_EPOCHS = 50

TIMESTEPS = 200
BETA_START = 1e-4
BETA_END = 0.02

PIXEL_LR = 2e-4
AE_LR = 1e-3
LATENT_LR = 2e-4

LATENT_CHANNELS = 4
LATENT_SIZE = 8

N_EVAL_SAMPLES = 5000
SAVE_SAMPLE_GRID = 64

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Seeds:", SEEDS)

Device: cuda
Seeds: [2, 22]


## 3. Caminhos

In [ ]:


def find_project_root(start: Path, markers: list[str], max_levels: int = 6) -> Path:
    """Sobe na árvore de pastas até encontrar uma pasta que contenha todos os markers."""
    path = start
    for _ in range(max_levels):
        if all((path / m).exists() for m in markers):
            return path
        path = path.parent
    raise RuntimeError(
        f"PROJECT_ROOT não encontrado a partir de {start}.\n"
        "Estrutura esperada: pasta com 'ArtBench-10/' e 'TP1-alunos-src-only/'."
    )

PROJECT_ROOT      = find_project_root(Path(".").resolve(), ["ArtBench-10", "TP1-alunos-src-only"])
SCRIPTS_DIR       = PROJECT_ROOT / "TP1-alunos-src-only" / "scripts"
KAGGLE_ROOT       = PROJECT_ROOT / "ArtBench-10"
TRAINING_CSV_PATH = PROJECT_ROOT / "TP1-alunos-src-only" / "student_start_pack" / "training_20_percent.csv"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
PIXEL_RESULTS_DIR = RESULTS_DIR / "pixel_diffusion"
LATENT_RESULTS_DIR = RESULTS_DIR / "latent_diffusion"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PIXEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LATENT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT   =", PROJECT_ROOT)
print("SCRIPTS_DIR    =", SCRIPTS_DIR)
print("KAGGLE_ROOT    =", KAGGLE_ROOT)
print("TRAINING CSV   =", TRAINING_CSV_PATH)

assert SCRIPTS_DIR.exists(), f"Não existe: {SCRIPTS_DIR}"
assert KAGGLE_ROOT.exists(), f"Não existe: {KAGGLE_ROOT}"
assert TRAINING_CSV_PATH.exists(), f"Não existe: {TRAINING_CSV_PATH}"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

PROJECT_ROOT   = C:\Users\Jose\Mestrado\IAG_TP1
SCRIPTS_DIR    = C:\Users\Jose\Mestrado\IAG_TP1\TP1-alunos-src-only\scripts
KAGGLE_ROOT    = C:\Users\Jose\Mestrado\IAG_TP1\ArtBench-10
TRAINING CSV   = C:\Users\Jose\Mestrado\IAG_TP1\TP1-alunos-src-only\student_start_pack\training_20_percent.csv


## 4. Dataset ArtBench subset 20%

In [3]:
from artbench_local_dataset import load_kaggle_artbench10_splits

hf_ds = load_kaggle_artbench10_splits(KAGGLE_ROOT)
train_hf = hf_ds["train"]
class_names = list(train_hf.features["label"].names)

print("Train size:", len(train_hf))
print("Número de classes:", len(class_names))
print("Classes:", class_names)

ModuleNotFoundError: No module named 'artbench_local_dataset'

In [8]:
transform = T.Compose([
    T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

class HFDatasetTorch(Dataset):
    def __init__(self, hf_split, transform=None, indices=None):
        self.ds = hf_split
        self.transform = transform
        self.indices = list(range(len(hf_split))) if indices is None else list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        ex = self.ds[real_idx]
        img = ex["image"]
        y = int(ex["label"])
        x = self.transform(img) if self.transform else img
        return x, y, real_idx

def load_ids_from_training_csv(csv_path: Path, index_column: str = "train_id_original") -> list[int]:
    ids = []
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        if index_column not in (reader.fieldnames or []):
            raise ValueError(f"Coluna {index_column!r} não encontrada. Disponíveis: {reader.fieldnames}")
        for row in reader:
            v = str(row.get(index_column, "")).strip()
            if v:
                ids.append(int(v))
    if len(ids) == 0:
        raise ValueError("Não foram lidos IDs do CSV.")
    return ids

def denorm(x):
    return (x * 0.5 + 0.5).clamp(0, 1)

train_ids = load_ids_from_training_csv(TRAINING_CSV_PATH, "train_id_original")
artbench_subset = HFDatasetTorch(train_hf, transform=transform, indices=train_ids)
artbench_loader = DataLoader(
    artbench_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Subset size:", len(artbench_subset))
print("Batches:", len(artbench_loader))

Subset size: 10000
Batches: 157


## 5. Helpers de avaliação

In [9]:
def collect_real_images(dataloader, n_samples=5000):
    real_batches = []
    total = 0
    for x, _, _ in dataloader:
        x = denorm(x.cpu())
        real_batches.append(x)
        total += x.size(0)
        if total >= n_samples:
            break
    return torch.cat(real_batches, dim=0)[:n_samples]

def to_uint8(x):
    return (x * 255).clamp(0, 255).to(torch.uint8)

def compute_fid_kid(real_images, fake_images, device, seed):
    real_uint8 = to_uint8(real_images)
    fake_uint8 = to_uint8(fake_images)

    fid = FrechetInceptionDistance(feature=2048).to(device)
    kid = KernelInceptionDistance(subset_size=100).to(device)

    fid.update(real_uint8.to(device), real=True)
    fid.update(fake_uint8.to(device), real=False)
    fid_value = float(fid.compute().item())

    kid.update(real_uint8.to(device), real=True)
    kid.update(fake_uint8.to(device), real=False)
    kid_mean, kid_std = kid.compute()

    return {
        "fid": float(fid_value),
        "kid_mean": float(kid_mean.item()),
        "kid_std": float(kid_std.item()),
        "seed": int(seed),
    }

real_images_eval = collect_real_images(artbench_loader, n_samples=N_EVAL_SAMPLES)
print("Real images for evaluation:", real_images_eval.shape)

Real images for evaluation: torch.Size([5000, 3, 32, 32])


## 6. Gaussian Diffusion helpers

In [10]:
class GaussianDiffusion:
    def __init__(self, num_timesteps=200, beta_start=1e-4, beta_end=0.02, device='cpu'):
        self.num_timesteps = num_timesteps
        self.device = device

        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = torch.cat(
            [torch.tensor([1.0], device=device), self.alphas_cumprod[:-1]], dim=0
        )

        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.posterior_variance = self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)

    def _get_index(self, vals, t, x_shape):
        out = vals.gather(-1, t)
        return out.reshape(t.shape[0], *((1,) * (len(x_shape) - 1)))

    def q_sample(self, x_0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_0)

        sqrt_alpha_prod = self._get_index(self.sqrt_alphas_cumprod, t, x_0.shape)
        sqrt_one_minus_alpha_prod = self._get_index(self.sqrt_one_minus_alphas_cumprod, t, x_0.shape)
        return sqrt_alpha_prod * x_0 + sqrt_one_minus_alpha_prod * noise

    @torch.no_grad()
    def p_sample(self, model, x, t, t_index):
        betas_t = self._get_index(self.betas, t, x.shape)
        sqrt_one_minus_alpha_cumprod_t = self._get_index(self.sqrt_one_minus_alphas_cumprod, t, x.shape)
        sqrt_recip_alphas_t = 1. / torch.sqrt(self._get_index(self.alphas, t, x.shape))

        predicted_noise = model(x, t)
        model_mean = sqrt_recip_alphas_t * (x - betas_t * predicted_noise / sqrt_one_minus_alpha_cumprod_t)

        if t_index == 0:
            return model_mean
        else:
            posterior_variance_t = self._get_index(self.posterior_variance, t, x.shape)
            noise = torch.randn_like(x)
            return model_mean + torch.sqrt(posterior_variance_t) * noise

    @torch.no_grad()
    def p_sample_loop(self, model, shape):
        model.eval()
        x = torch.randn(shape, device=self.device)

        for i in reversed(range(self.num_timesteps)):
            t = torch.full((shape[0],), i, device=self.device, dtype=torch.long)
            x = self.p_sample(model, x, t, i)

        return x

In [11]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

class ResnetBlock(nn.Module):
    def __init__(self, dim, time_emb_dim, out_dim=None):
        super().__init__()
        self.out_dim = out_dim or dim
        self.mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, self.out_dim))
        self.conv1 = nn.Conv2d(dim, self.out_dim, 3, padding=1)
        self.conv2 = nn.Conv2d(self.out_dim, self.out_dim, 3, padding=1)
        self.norm1 = nn.GroupNorm(4, dim)
        self.norm2 = nn.GroupNorm(4, self.out_dim)
        self.act = nn.SiLU()
        self.shortcut = nn.Conv2d(dim, self.out_dim, 1) if dim != self.out_dim else nn.Identity()

    def forward(self, x, time_emb):
        h = self.norm1(x)
        h = self.act(h)
        h = self.conv1(h)
        time_emb = self.mlp(time_emb)
        h = h + time_emb[:, :, None, None]
        h = self.norm2(h)
        h = self.act(h)
        h = self.conv2(h)
        return self.shortcut(x) + h

## 7. Models

In [12]:
class PixelUNet(nn.Module):
    def __init__(self, in_channels=3, model_channels=64, num_res_blocks=3):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPosEmb(model_channels),
            nn.Linear(model_channels, model_channels * 4),
            nn.SiLU(),
            nn.Linear(model_channels * 4, model_channels * 4),
        )

        self.init_conv = nn.Conv2d(in_channels, model_channels, 3, padding=1)
        self.res_blocks = nn.ModuleList([
            ResnetBlock(model_channels, model_channels * 4)
            for _ in range(num_res_blocks)
        ])
        self.out_conv = nn.Conv2d(model_channels, in_channels, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        h = self.init_conv(x)
        for block in self.res_blocks:
            h = block(h, t_emb)
        return self.out_conv(h)

class Encoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.mu = nn.Conv2d(64, latent_channels, kernel_size=1)
        self.logvar = nn.Conv2d(64, latent_channels, kernel_size=1)

    def forward(self, x):
        h = self.net(x)
        return self.mu(h), self.logvar(h)

class Decoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.initial_conv = nn.Conv2d(latent_channels, 64, kernel_size=1)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(64, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z):
        h = self.initial_conv(z)
        return self.net(h)

class VAE(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.encoder = Encoder(latent_channels)
        self.decoder = Decoder(latent_channels)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

class LatentDenoiseNetwork(nn.Module):
    def __init__(self, latent_channels=4, model_channels=64, num_res_blocks=3):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPosEmb(model_channels),
            nn.Linear(model_channels, model_channels * 4),
            nn.SiLU(),
            nn.Linear(model_channels * 4, model_channels * 4),
        )

        self.init_conv = nn.Conv2d(latent_channels, model_channels, 3, padding=1)
        self.res_blocks = nn.ModuleList([
            ResnetBlock(model_channels, model_channels * 4)
            for _ in range(num_res_blocks)
        ])
        self.out_conv = nn.Conv2d(model_channels, latent_channels, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        h = self.init_conv(x)
        for block in self.res_blocks:
            h = block(h, t_emb)
        return self.out_conv(h)

## 8. Training helpers

In [13]:
def train_autoencoder(model, loader, epochs=20, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    model.train()

    for epoch in range(epochs):
        running = 0.0
        n_batches = 0

        for x, _, _ in loader:
            x = x.to(device)
            recon, mu, logvar = model(x)
            recon_loss = F.mse_loss(recon, x, reduction='mean')
            kld_loss = torch.mean(-0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp(), dim=(1,2,3)))
            loss = recon_loss + 0.01 * kld_loss

            opt.zero_grad()
            loss.backward()
            opt.step()

            running += loss.item()
            n_batches += 1

        avg = running / max(n_batches, 1)
        history.append(avg)
        print(f'AE epoch {epoch + 1:02d}/{epochs} | loss: {avg:.4f}')

    return history

def train_diffusion(model, loader, schedule, epochs=20, lr=2e-4, encode_fn=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    model.train()

    for epoch in range(epochs):
        running = 0.0
        n_batches = 0

        for x, _, _ in loader:
            x = x.to(device)
            if encode_fn is not None:
                with torch.no_grad():
                    x = encode_fn(x)

            batch_size = x.shape[0]
            t = torch.randint(0, schedule.num_timesteps, (batch_size,), device=device).long()
            noise = torch.randn_like(x)

            x_t = schedule.q_sample(x, t, noise)
            pred_noise = model(x_t, t)

            loss = F.mse_loss(pred_noise, noise)

            opt.zero_grad()
            loss.backward()
            opt.step()

            running += loss.item()
            n_batches += 1

        avg = running / max(n_batches, 1)
        history.append(avg)
        print(f'Diff epoch {epoch + 1:02d}/{epochs} | loss: {avg:.4f}')

    return history

def save_summary(df, out_json, model_type):
    summary = {
        "model_type": model_type,
        "n_runs": int(len(df)),
        "fid_mean": float(df["fid"].mean()),
        "fid_std": float(df["fid"].std(ddof=1)) if len(df) > 1 else 0.0,
        "kid_mean_mean": float(df["kid_mean"].mean()),
        "kid_mean_std": float(df["kid_mean"].std(ddof=1)) if len(df) > 1 else 0.0,
        "kid_std_mean": float(df["kid_std"].mean()),
    }
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=4, ensure_ascii=False)
    return summary

## 9. Multiple-seed experiment loop

In [14]:
pixel_results = []
latent_results = []

for seed in SEEDS:
    print("\n" + "="*60)
    print(f"Running seed {seed}")
    print("="*60)

    set_seed(seed)

    # -----------------------------
    # Pixel diffusion
    # -----------------------------
    pixel_diffusion = GaussianDiffusion(
        num_timesteps=TIMESTEPS,
        beta_start=BETA_START,
        beta_end=BETA_END,
        device=device
    )
    pixel_model = PixelUNet(in_channels=3, model_channels=64).to(device)

    _ = train_diffusion(
        model=pixel_model,
        loader=artbench_loader,
        schedule=pixel_diffusion,
        epochs=PIXEL_EPOCHS,
        lr=PIXEL_LR
    )

    sampled_pixels = pixel_diffusion.p_sample_loop(pixel_model, shape=(SAVE_SAMPLE_GRID, 3, 32, 32))
    pixel_grid = make_grid(denorm(sampled_pixels[:64].cpu()), nrow=8, padding=2)
    save_image(pixel_grid, PIXEL_RESULTS_DIR / f"grid_seed_{seed}.png")

    pixel_fake_images = []
    total = 0
    while total < N_EVAL_SAMPLES:
        curr = min(BATCH_SIZE, N_EVAL_SAMPLES - total)
        samples = pixel_diffusion.p_sample_loop(pixel_model, shape=(curr, 3, 32, 32))
        pixel_fake_images.append(denorm(samples.cpu()))
        total += curr
        print(f"Pixel generated: {total}/{N_EVAL_SAMPLES}")

    pixel_fake_images = torch.cat(pixel_fake_images, dim=0)
    pixel_metrics = compute_fid_kid(real_images_eval, pixel_fake_images, device, seed)
    pixel_metrics["model_type"] = "pixel_diffusion"
    pixel_results.append(pixel_metrics)

    with open(PIXEL_RESULTS_DIR / f"metrics_seed_{seed}.json", "w", encoding="utf-8") as f:
        json.dump({"metrics": pixel_metrics}, f, indent=4, ensure_ascii=False)

    pd.DataFrame([pixel_metrics]).to_csv(PIXEL_RESULTS_DIR / f"metrics_seed_{seed}.csv", index=False)
    torch.save(pixel_model.state_dict(), CHECKPOINT_DIR / f"pixel_diffusion_artbench_subset20_seed{seed}.pth")

    # -----------------------------
    # Latent diffusion
    # -----------------------------
    vae = VAE(latent_channels=LATENT_CHANNELS).to(device)
    _ = train_autoencoder(vae, artbench_loader, epochs=AE_EPOCHS, lr=AE_LR)

    for p in vae.parameters():
        p.requires_grad = False
    vae.eval()

    latent_diffusion = GaussianDiffusion(
        num_timesteps=TIMESTEPS,
        beta_start=BETA_START,
        beta_end=BETA_END,
        device=device
    )
    latent_model = LatentDenoiseNetwork(
        latent_channels=LATENT_CHANNELS,
        model_channels=64,
        num_res_blocks=3
    ).to(device)

    def encode_fn(x):
        mu, logvar = vae.encoder(x)
        return vae.reparameterize(mu, logvar)

    _ = train_diffusion(
        model=latent_model,
        loader=artbench_loader,
        schedule=latent_diffusion,
        epochs=LATENT_EPOCHS,
        lr=LATENT_LR,
        encode_fn=encode_fn
    )

    z_sampled = latent_diffusion.p_sample_loop(latent_model, shape=(SAVE_SAMPLE_GRID, LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE))
    with torch.no_grad():
        recon = vae.decoder(z_sampled)

    latent_grid = make_grid(denorm(recon[:64].cpu()), nrow=8, padding=2)
    ##save_image(latent_grid, LATENT_RESULTS_DIR / f"grid_seed_{seed}.png")

    latent_fake_images = []
    total = 0
    while total < N_EVAL_SAMPLES:
        curr = min(BATCH_SIZE, N_EVAL_SAMPLES - total)
        z = latent_diffusion.p_sample_loop(latent_model, shape=(curr, LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE))
        with torch.no_grad():
            x = vae.decoder(z)
        latent_fake_images.append(denorm(x.cpu()))
        total += curr
        print(f"Latent generated: {total}/{N_EVAL_SAMPLES}")

    latent_fake_images = torch.cat(latent_fake_images, dim=0)
    latent_metrics = compute_fid_kid(real_images_eval, latent_fake_images, device, seed)
    latent_metrics["model_type"] = "latent_diffusion"
    latent_results.append(latent_metrics)

    with open(LATENT_RESULTS_DIR / f"metrics_seed_{seed}.json", "w", encoding="utf-8") as f:
        json.dump({"metrics": latent_metrics}, f, indent=4, ensure_ascii=False)

    pd.DataFrame([latent_metrics]).to_csv(LATENT_RESULTS_DIR / f"metrics_seed_{seed}.csv", index=False)
    torch.save(vae.state_dict(), CHECKPOINT_DIR / f"vae_for_latent_diffusion_artbench_subset20_seed{seed}.pth")
    torch.save(latent_model.state_dict(), CHECKPOINT_DIR / f"latent_diffusion_artbench_subset20_seed{seed}.pth")


Running seed 2
Diff epoch 01/50 | loss: 0.2397
Diff epoch 02/50 | loss: 0.1320
Diff epoch 03/50 | loss: 0.1116
Diff epoch 04/50 | loss: 0.1076
Diff epoch 05/50 | loss: 0.1018
Diff epoch 06/50 | loss: 0.1016
Diff epoch 07/50 | loss: 0.1021
Diff epoch 08/50 | loss: 0.0955
Diff epoch 09/50 | loss: 0.0978
Diff epoch 10/50 | loss: 0.0969
Diff epoch 11/50 | loss: 0.0952
Diff epoch 12/50 | loss: 0.0939
Diff epoch 13/50 | loss: 0.0948
Diff epoch 14/50 | loss: 0.0947
Diff epoch 15/50 | loss: 0.0935
Diff epoch 16/50 | loss: 0.0914
Diff epoch 17/50 | loss: 0.0915
Diff epoch 18/50 | loss: 0.0925
Diff epoch 19/50 | loss: 0.0908
Diff epoch 20/50 | loss: 0.0915


KeyboardInterrupt: 

## 10. Aggregate results

In [ ]:
pixel_df = pd.DataFrame(pixel_results).sort_values("fid").reset_index(drop=True)
latent_df = pd.DataFrame(latent_results).sort_values("fid").reset_index(drop=True)

pixel_df.to_csv(PIXEL_RESULTS_DIR / "all_seeds_results.csv", index=False)
latent_df.to_csv(LATENT_RESULTS_DIR / "all_seeds_results.csv", index=False)

pixel_summary = save_summary(pixel_df, PIXEL_RESULTS_DIR / "summary.json", "pixel_diffusion")
latent_summary = save_summary(latent_df, LATENT_RESULTS_DIR / "summary.json", "latent_diffusion")

print("Pixel summary:", pixel_summary)
print("Latent summary:", latent_summary)

## 11. Final comparison

In [ ]:
comparison_df = pd.DataFrame([
    {
        "model_type": "pixel_diffusion",
        "fid_mean": pixel_summary["fid_mean"],
        "fid_std": pixel_summary["fid_std"],
        "kid_mean_mean": pixel_summary["kid_mean_mean"],
        "kid_mean_std": pixel_summary["kid_mean_std"],
    },
    {
        "model_type": "latent_diffusion",
        "fid_mean": latent_summary["fid_mean"],
        "fid_std": latent_summary["fid_std"],
        "kid_mean_mean": latent_summary["kid_mean_mean"],
        "kid_mean_std": latent_summary["kid_mean_std"],
    }
]).sort_values("fid_mean").reset_index(drop=True)

comparison_df

In [ ]:
best_model = comparison_df.iloc[0]
print("=== Melhor diffusion model ===")
print(best_model)

## 12. O que fica guardado

### Pixel diffusion
- `results/pixel_diffusion/grid_seed_<seed>.png`
- `results/pixel_diffusion/metrics_seed_<seed>.json`
- `results/pixel_diffusion/metrics_seed_<seed>.csv`
- `results/pixel_diffusion/all_seeds_results.csv`
- `results/pixel_diffusion/summary.json`

### Latent diffusion
- `results/latent_diffusion/grid_seed_<seed>.png`
- `results/latent_diffusion/metrics_seed_<seed>.json`
- `results/latent_diffusion/metrics_seed_<seed>.csv`
- `results/latent_diffusion/all_seeds_results.csv`
- `results/latent_diffusion/summary.json`

### Checkpoints
- um checkpoint por seed para pixel diffusion
- um VAE por seed
- um latent diffusion model por seed